In [20]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

BASE_URL = "https://etrain.info/list/EXP-TRAINS?page={}"

headers = {
    "User-Agent": "Mozilla/5.0"
}

data = []

for page in range(1, 103):
    print(f"Scraping page {page}/102")

    url = BASE_URL.format(page)
    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print(f"Failed page {page}")
        continue

    soup = BeautifulSoup(response.text, "html.parser")

    table = soup.find("table")

    if table is None:
        print(f"No table on page {page}")
        continue

    rows = table.find_all("tr")

    for row in rows[1:]:
        cols = [x.get_text(strip=True) for x in row.find_all("td")]

        if len(cols) >= 4:
            data.append({
                "Train Number": cols[0],
                "Train Name": cols[1],
                "Starting Station": cols[2],
                "Terminating Station": cols[3]
            })

    time.sleep(0.5)      # be polite to the server

df = pd.DataFrame(data)

df.drop_duplicates(inplace=True)

df.to_csv("mail.csv", index=False)

print(df.head())
print(f"\nTotal trains: {len(df)}")
print("Saved as mail.csv")

Scraping page 1/102
Scraping page 2/102
Scraping page 3/102
Scraping page 4/102
Scraping page 5/102
Scraping page 6/102
Scraping page 7/102
Scraping page 8/102
Scraping page 9/102
Scraping page 10/102
Scraping page 11/102
Scraping page 12/102
Scraping page 13/102
Scraping page 14/102
Scraping page 15/102
Scraping page 16/102
Scraping page 17/102
Scraping page 18/102
Scraping page 19/102
Scraping page 20/102
Scraping page 21/102
Scraping page 22/102
Scraping page 23/102
Scraping page 24/102
Scraping page 25/102
Scraping page 26/102
Scraping page 27/102
Scraping page 28/102
Scraping page 29/102
Scraping page 30/102
Scraping page 31/102
Scraping page 32/102
Scraping page 33/102
Scraping page 34/102
Scraping page 35/102
Scraping page 36/102
Scraping page 37/102
Scraping page 38/102
Scraping page 39/102
Scraping page 40/102
Scraping page 41/102
Scraping page 42/102
Scraping page 43/102
Scraping page 44/102
Scraping page 45/102
Scraping page 46/102
Scraping page 47/102
Scraping page 48/102
S

In [18]:
import httpx
import pandas as pd
from bs4 import BeautifulSoup
import asyncio

BASE_URL = "https://etrain.info/list/STATIONS?page={}"

headers = {
    "User-Agent": "Mozilla/5.0"
}

# Adjust if you get rate-limited
sem = asyncio.Semaphore(20)


async def scrape_page(client, page):
    async with sem:
        try:
            r = await client.get(BASE_URL.format(page), timeout=20)
            r.raise_for_status()

            soup = BeautifulSoup(r.text, "lxml")

            table = soup.find("table")
            if table is None:
                print(f"Page {page}: No table found")
                return []

            stations = []

            rows = table.find_all("tr")[1:]

            for row in rows:
                cols = row.find_all("td")

                if len(cols) >= 2:
                    stations.append({
                        "station_code": cols[0].get_text(strip=True),
                        "station_name": cols[1].get_text(strip=True)
                    })

            print(f"✓ Page {page}")

            return stations

        except Exception as e:
            print(f"✗ Page {page}: {e}")
            return []


async def main():

    limits = httpx.Limits(
        max_connections=50,
        max_keepalive_connections=20
    )

    async with httpx.AsyncClient(
        headers=headers,
        limits=limits,
        follow_redirects=True,
        http2=True
    ) as client:

        tasks = [
            scrape_page(client, page)
            for page in range(1, 361)
        ]

        results = await asyncio.gather(*tasks)

    stations = []

    for result in results:
        stations.extend(result)

    df = pd.DataFrame(stations)

    df = (
        df
        .drop_duplicates()
        .sort_values("station_code")
        .reset_index(drop=True)
    )

    df.to_csv("stations.csv", index=False)

    print("\nFinished!")
    print("Total Stations:", len(df))
    print(df.head())


# Run in Jupyter/Colab/Kaggle
await main()

/usr/local/lib/python3.12/dist-packages/httpcore/_sync/__init__.py:2: RuntimeWarning: coroutine 'main' was never awaited
  from .connection_pool import ConnectionPool


✓ Page 1
✓ Page 21
✓ Page 2
✓ Page 22
✓ Page 23
✓ Page 24
✓ Page 25
✓ Page 3
✓ Page 26
✓ Page 28
✓ Page 27
✓ Page 4
✓ Page 29
✓ Page 30
✓ Page 32
✓ Page 31
✓ Page 33
✓ Page 34
✓ Page 5
✓ Page 36
✓ Page 35
✓ Page 37
✓ Page 39
✓ Page 38
✓ Page 40
✓ Page 41
✓ Page 6
✓ Page 42
✓ Page 43
✓ Page 44
✓ Page 45
✓ Page 47
✓ Page 46
✓ Page 48
✓ Page 49
✓ Page 50
✓ Page 51
✓ Page 7
✓ Page 52
✓ Page 54
✓ Page 53
✓ Page 55
✓ Page 56
✓ Page 57
✓ Page 58
✓ Page 60
✓ Page 61
✓ Page 59
✓ Page 8
✓ Page 64
✓ Page 62
✓ Page 63
✓ Page 65
✓ Page 66
✓ Page 68
✓ Page 69
✓ Page 67
✓ Page 71
✓ Page 70
✓ Page 72
✓ Page 11
✓ Page 73
✓ Page 74
✓ Page 76
✓ Page 75
✓ Page 77
✓ Page 80
✓ Page 81
✓ Page 79
✓ Page 78
✓ Page 82
✓ Page 83
✓ Page 84
✓ Page 85
✓ Page 86
✓ Page 9
✓ Page 87
✓ Page 90
✓ Page 89
✓ Page 88
✓ Page 92
✓ Page 91
✓ Page 93
✓ Page 94
✓ Page 95
✓ Page 96
✓ Page 97
✓ Page 98
✓ Page 99
✓ Page 100
✓ Page 12
✓ Page 101
✓ Page 13
✓ Page 18
✓ Page 10
✓ Page 102
✓ Page 107
✓ Page 104
✓ Page 108
✓ Page 109
✓ 